# Ficha CNN RGB Multiclass — CIFAR-10

Notebook de resolução/base para a ficha **Classificação de imagens RGB com CNNs usando CIFAR-10**.

O notebook cobre:
- T1 a T6: dataset, preprocessing, data loaders, visualização e balanceamento;
- T7: definição dos 5 modelos;
- T8: treino dos modelos;
- T9: avaliação, relatório de classificação e matriz de confusão;
- T10: uso dos modelos para prever uma imagem;
- T11: comparação final e preparação dos resultados para entrega.

> Antes de executar: idealmente usa **Google Colab com GPU** ou uma máquina com CUDA. Em CPU, o treino completo pode demorar bastante.

## 0. Instalações e imports

Executa esta célula primeiro. Ela tenta instalar apenas os pacotes que estiverem em falta.

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    """Instala o pacote apenas se o import não existir."""
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

# Torch costuma já vir instalado no Colab. Se não vier, instala.
ensure_package("torch", "torch torchvision torchaudio")
ensure_package("torchinfo", "torchinfo")
ensure_package("sklearn", "scikit-learn")
ensure_package("PIL", "pillow")
ensure_package("gdown", "gdown")
ensure_package("pandas", "pandas")
ensure_package("matplotlib", "matplotlib")

print("Dependências verificadas.")

In [ ]:
import os
import random
import time
import tarfile
import zipfile
import shutil
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torch.nn as nn
from torch.nn import (
    BatchNorm2d, Dropout2d, Sequential, Linear, Conv2d,
    MaxPool2d, ReLU, Softmax, Module, CrossEntropyLoss
)
from torch.optim import SGD
from torch.nn.init import kaiming_uniform_, xavier_uniform_

from torchinfo import summary

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 0.1. Constantes

A ficha pede `batch_size = 128`.

In [ ]:
PATH = Path("./cifar")
PATH_CLASSES = PATH / "labels.txt"
PATH_TRAIN = PATH / "train"
PATH_TEST = PATH / "test"

BATCH_SIZE = 128
RESULTS_DIR = Path("resultados")
RESULTS_DIR.mkdir(exist_ok=True)

CIFAR_URL = "http://pjreddie.com/media/files/cifar.tgz"
LOCAL_DATASET_ZIP_PATTERNS = [
    "notebooksDataset(1).zip",
    "notebooksDataset.zip",
    "*.zip"
]

MODEL_FILES = {
    "ResNet": "CNNModel_cifar_Resnet.pth",
    "CNN1": "CNNModel_cifar_1.pth",
    "CNN2": "CNNModel_cifar_2.pth",
    "CNN3": "CNNModel_cifar_3.pth",
    "CNN4": "CNNModel_cifar_4.pth",
}

GDRIVE_MODEL_IDS = {
    "ResNet": "1pg3nKSWsttMlaH7APwgfkgAUlBBlUNw_",
    "CNN1": "1YiY4zmuaQGwk_mSjK3LNWUMJfAABkbCx",
    "CNN2": "1p4Udbjb4q3uerAk_F8Jxs7H2UZL4QUN",
    "CNN3": "18KNK0wgJZjCVmgWB5ykUAVGwaU5PcURU",
    "CNN4": "1VWCisHcM1K63-6qYeLjNqgiTcTBmPy_1",
}

## 1. Preparar o dataset

Esta célula faz uma destas opções:

1. Se já existir a pasta `cifar/`, usa-a.
2. Se tiveres o ficheiro `notebooksDataset(1).zip` na mesma pasta do notebook, extrai o `cifar.tgz` e os modelos `.pth`.
3. Se não houver zip local, descarrega o `cifar.tgz` da internet.

No Colab: faz upload do `notebooksDataset(1).zip` para a pasta do notebook, ou deixa a célula descarregar o dataset.

In [ ]:
def find_local_zip():
    for pattern in LOCAL_DATASET_ZIP_PATTERNS:
        for p in Path(".").glob(pattern):
            if p.is_file() and "notebooksDataset" in p.name:
                return p
    # fallback: qualquer zip que tenha cifar.tgz lá dentro
    for p in Path(".").glob("*.zip"):
        try:
            with zipfile.ZipFile(p) as z:
                if any(name.endswith("cifar.tgz") for name in z.namelist()):
                    return p
        except zipfile.BadZipFile:
            pass
    return None

def extract_from_dataset_zip(zip_path):
    print(f"A extrair ficheiros de {zip_path}...")
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()

        # Extrair cifar.tgz
        cifar_members = [n for n in names if n.endswith("cifar.tgz")]
        if cifar_members and not Path("cifar.tgz").exists():
            member = cifar_members[0]
            z.extract(member, ".")
            extracted = Path(member)
            if extracted != Path("cifar.tgz"):
                shutil.move(str(extracted), "cifar.tgz")

        # Extrair modelos .pth, se existirem
        for model_file in MODEL_FILES.values():
            members = [n for n in names if n.endswith(model_file)]
            if members and not Path(model_file).exists():
                member = members[0]
                z.extract(member, ".")
                extracted = Path(member)
                if extracted != Path(model_file):
                    shutil.move(str(extracted), model_file)

def ensure_cifar_dataset():
    if PATH_TRAIN.exists() and PATH_TEST.exists() and PATH_CLASSES.exists():
        print("Dataset já existe em ./cifar/")
        return

    zip_path = find_local_zip()
    if zip_path:
        extract_from_dataset_zip(zip_path)

    if not Path("cifar.tgz").exists():
        print("A descarregar cifar.tgz...")
        urllib.request.urlretrieve(CIFAR_URL, "cifar.tgz")

    print("A extrair cifar.tgz...")
    with tarfile.open("cifar.tgz", "r:gz") as tar:
        tar.extractall(".")

    print("Dataset pronto:", PATH.resolve())

ensure_cifar_dataset()
print("Train existe?", PATH_TRAIN.exists())
print("Test existe?", PATH_TEST.exists())
print("Labels existe?", PATH_CLASSES.exists())

## 2. Preparação dos dados

Inclui:
- lista de classes;
- normalização;
- mudança do canal de cores para formato compatível com CNN: `(3, 32, 32)`;
- criação dos `DataLoader` com holdout: 80% treino e 20% validação.

In [ ]:
def get_classes(path):
    with open(path, encoding="utf-8") as fich_labels:
        labels = fich_labels.read().split()
        classes = dict(zip(labels, list(range(len(labels)))))
    return classes

dic_classes = get_classes(PATH_CLASSES)
idx_to_class = {v: k for k, v in dic_classes.items()}

print(dic_classes)
print(idx_to_class)

In [ ]:
def preprocessar(imagem):
    """Pré-processamento usado na aula: normalização e conversão do formato da imagem.

    A imagem vem em H/W/C e fica com shape (3, 32, 32), adequado para Conv2D.
    Mantém-se a transposição da aula: transpose(2, 1, 0).
    Como as imagens CIFAR têm 32x32, o shape final continua correto.
    """
    imagem = np.array(imagem)

    cifar_mean = np.array([0.4914, 0.4822, 0.4465]).reshape(1, 1, -1)
    cifar_std  = np.array([0.2023, 0.1994, 0.2010]).reshape(1, 1, -1)

    imagem = (imagem - cifar_mean) / cifar_std

    xmax, xmin = imagem.max(), imagem.min()
    imagem = (imagem - xmin) / (xmax - xmin + 1e-8)

    imagem = imagem.transpose(2, 1, 0)
    return imagem.astype(np.float32)

class Cifar10Dataset(Dataset):
    def __init__(self, path, num_imagens=0, transforms=None):
        files = sorted(os.listdir(path))
        files = [os.path.join(path, f) for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))]

        if num_imagens == 0:
            num_imagens = len(files)

        self.num_imagens = num_imagens
        self.files = random.sample(files, self.num_imagens) if num_imagens < len(files) else files
        self.transforms = transforms
        self.labels = [self.label_from_file(f) for f in self.files]

    def label_from_file(self, fich_imagem):
        label_classe = Path(fich_imagem).stem.split("_")[-1]
        return dic_classes[label_classe]

    def __len__(self):
        return self.num_imagens

    def __getitem__(self, idx):
        fich_imagem = self.files[idx]
        imagem = Image.open(fich_imagem).convert("RGB")
        imagem = preprocessar(imagem)
        label = self.labels[idx]

        if self.transforms:
            imagem = self.transforms(imagem)

        return torch.tensor(imagem, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [ ]:
def prepare_data_loaders(path_train, path_test):
    dataset_train = Cifar10Dataset(path_train, transforms=None)
    dataset_test = Cifar10Dataset(path_test, transforms=None)

    train_size = int(0.8 * len(dataset_train))
    val_size = len(dataset_train) - train_size

    train, validation = random_split(
        dataset_train,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_dl = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(validation, batch_size=BATCH_SIZE, shuffle=False)
    test_dl = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

    return train_dl, val_dl, test_dl, dataset_train, dataset_test, train, validation

train_dl, val_dl, test_dl, dataset_train, dataset_test, train_subset, val_subset = prepare_data_loaders(PATH_TRAIN, PATH_TEST)

print(f"Quantidade de casos de Treino: {len(train_dl.dataset)}")
print(f"Quantidade de casos de Validação: {len(val_dl.dataset)}")
print(f"Quantidade de casos de Teste: {len(test_dl.dataset)}")

x, y = next(iter(train_dl))
print(f"Shape batch treino, input: {x.shape}, output: {y.shape}")
print(f"Valor máximo: {torch.max(x):.4f} | Valor mínimo: {torch.min(x):.4f}")
print("Primeiras labels:", y[:20])

## 3. Visualizar os dados

Mostra um batch de imagens de treino com os labels por extenso.

In [ ]:
def output_label(label):
    if isinstance(label, torch.Tensor):
        label = label.item()
    return idx_to_class[int(label)]

def tensor_to_image(img_tensor):
    # Inverso da transposição usada na aula: C/W/H -> H/W/C visualizável
    img = img_tensor.detach().cpu()
    return img.permute(2, 1, 0).numpy()

def visualize_images(dl, n=25):
    inputs, targets = next(iter(dl))
    n = min(n, len(inputs))

    cols = 5
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(10, 10))

    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        plt.axis("off")
        plt.imshow(tensor_to_image(inputs[i]))
        plt.title(output_label(targets[i]), fontsize=9)

    plt.tight_layout()
    plt.show()

visualize_images(train_dl, n=25)

## 4. Verificar balanceamento do dataset

Inclui treino, validação e teste.

In [ ]:
def labels_from_dataset(ds):
    if isinstance(ds, Subset):
        base = ds.dataset
        return [base.labels[i] for i in ds.indices]
    if hasattr(ds, "labels"):
        return list(ds.labels)
    return [int(ds[i][1]) for i in range(len(ds))]

def plot_balance(labels, title):
    values, counts = np.unique(labels, return_counts=True)
    names = [idx_to_class[int(v)] for v in values]

    df = pd.DataFrame({"classe": names, "quantidade": counts})
    display(df)

    plt.figure(figsize=(10, 4))
    plt.bar(df["classe"], df["quantidade"])
    plt.title(title)
    plt.xlabel("Classe")
    plt.ylabel("Quantidade")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

print("Balanceamento — Treino")
plot_balance(labels_from_dataset(train_dl.dataset), "Balanceamento do conjunto de treino")

print("Balanceamento — Validação")
plot_balance(labels_from_dataset(val_dl.dataset), "Balanceamento do conjunto de validação")

print("Balanceamento — Teste")
plot_balance(labels_from_dataset(test_dl.dataset), "Balanceamento do conjunto de teste")

# 5. Definir os modelos

Serão definidos os modelos pedidos:
- ResNet;
- CNN 1;
- CNN 2;
- CNN 3;
- CNN 4.

## 5.1. Modelo ResNet

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(3, 3),
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=(3, 3),
            stride=1,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=(1, 1),
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = nn.ReLU()(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = nn.ReLU()(out)
        return out

class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=(3, 3),
            stride=1,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)

        self.block1 = self._create_block(64, 64, stride=1)
        self.block2 = self._create_block(64, 128, stride=2)
        self.block3 = self._create_block(128, 256, stride=2)
        self.block4 = self._create_block(256, 512, stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _create_block(self, in_channels, out_channels, stride):
        return nn.Sequential(
            ResidualBlock(in_channels, out_channels, stride),
            ResidualBlock(out_channels, out_channels, 1)
        )

    def forward(self, x):
        out = nn.ReLU()(self.bn1(self.conv1(x)))
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.block4(out)
        out = nn.AvgPool2d(4)(out)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

model = ResNet()
print(summary(model, input_size=(BATCH_SIZE, 3, 32, 32), verbose=0))

## 5.2. Modelo CNN 1

In [ ]:
class CNNModel_1(Module):
    def __init__(self):
        super(CNNModel_1, self).__init__()
        self.layer1 = Sequential(
            Conv2d(in_channels=3, out_channels=32, kernel_size=(3, 3)),
            ReLU(),
            MaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        )
        self.layer2 = Sequential(
            Conv2d(in_channels=32, out_channels=32, kernel_size=(3, 3)),
            ReLU(),
            MaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        )
        self.fc1 = Linear(in_features=32 * 6 * 6, out_features=100)
        kaiming_uniform_(self.fc1.weight, nonlinearity="relu")
        self.act1 = ReLU()
        self.fc2 = Linear(in_features=100, out_features=10)
        xavier_uniform_(self.fc2.weight)
        self.act2 = Softmax(dim=1)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.act1(out)
        out = self.fc2(out)
        out = self.act2(out)
        return out

model = CNNModel_1()
print(summary(model, input_size=(BATCH_SIZE, 3, 32, 32), verbose=0))

## 5.3. Modelo CNN 2

In [ ]:
class CNNModel_2(Module):
    def __init__(self):
        super(CNNModel_2, self).__init__()
        self.layer1 = Sequential(
            Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=0),
            ReLU(),
            MaxPool2d(kernel_size=2)
        )
        self.layer2 = Sequential(
            Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=0),
            ReLU(),
            MaxPool2d(kernel_size=2)
        )
        self.fc1 = Linear(in_features=32 * 6 * 6, out_features=10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        return out

model = CNNModel_2()
print(summary(model, input_size=(BATCH_SIZE, 3, 32, 32), verbose=0))

## 5.4. Modelo CNN 3

In [ ]:
class CNNModel_3(Module):
    def __init__(self):
        super(CNNModel_3, self).__init__()
        self.layer1 = nn.Sequential(
            Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            BatchNorm2d(32),
            ReLU(),
            MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = nn.Sequential(
            Conv2d(in_channels=32, out_channels=64, kernel_size=3),
            BatchNorm2d(64),
            ReLU(),
            MaxPool2d(2)
        )
        self.fc1 = Linear(in_features=64 * 7 * 7, out_features=600)
        self.drop = nn.Dropout2d(0.25)
        self.fc2 = Linear(in_features=600, out_features=120)
        self.fc3 = Linear(in_features=120, out_features=10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(-1, 64 * 7 * 7)
        out = self.fc1(out)
        out = self.drop(out)
        out = self.fc2(out)
        out = self.fc3(out)
        return out

model = CNNModel_3()
print(summary(model, input_size=(BATCH_SIZE, 3, 32, 32), verbose=0))

## 5.5. Modelo CNN 4

In [ ]:
class CNNModel_4(Module):
    def __init__(self):
        super(CNNModel_4, self).__init__()
        self.layer1 = Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=5, padding=0),
            BatchNorm2d(32),
            ReLU(),
            MaxPool2d(2),
            Dropout2d(0.2)
        )
        self.fc1 = Linear(in_features=32 * 14 * 14, out_features=128)
        self.fc2 = Linear(in_features=128, out_features=10)

    def forward(self, x):
        out = self.layer1(x)
        out = out.view(-1, 32 * 14 * 14)
        out = self.fc1(out)
        out = self.fc2(out)
        return out

model = CNNModel_4()
print(summary(model, input_size=(BATCH_SIZE, 3, 32, 32), verbose=0))

## 6. Configuração de treino dos modelos

A ficha pede:
- ResNet: 30 epochs;
- CNN 1, CNN 2 e CNN 3: 15 epochs;
- CNN 4: 75 epochs;
- `learning_rate = 0.001`;
- `CrossEntropyLoss`;
- `SGD`.

Se quiseres apenas testar rapidamente com os modelos `.pth` fornecidos, muda `RUN_TRAINING = False`.
Para a entrega, deixa `RUN_TRAINING = True` e executa tudo.

In [ ]:
MODEL_CONFIGS = {
    "ResNet": {
        "constructor": ResNet,
        "epochs": 30,
        "lr": 0.001,
        "file": MODEL_FILES["ResNet"],
        "architecture": "ResNet com blocos residuais, BatchNorm e Linear final"
    },
    "CNN1": {
        "constructor": CNNModel_1,
        "epochs": 15,
        "lr": 0.001,
        "file": MODEL_FILES["CNN1"],
        "architecture": "Conv + ReLU + MaxPool; Conv + ReLU + MaxPool; Linear + ReLU + Linear + Softmax"
    },
    "CNN2": {
        "constructor": CNNModel_2,
        "epochs": 15,
        "lr": 0.001,
        "file": MODEL_FILES["CNN2"],
        "architecture": "Conv + ReLU + MaxPool; Conv + ReLU + MaxPool; Linear"
    },
    "CNN3": {
        "constructor": CNNModel_3,
        "epochs": 15,
        "lr": 0.001,
        "file": MODEL_FILES["CNN3"],
        "architecture": "Conv + BatchNorm + ReLU + MaxPool; Conv + BatchNorm + ReLU + MaxPool; Linear + Dropout + Linear + Linear"
    },
    "CNN4": {
        "constructor": CNNModel_4,
        "epochs": 75,
        "lr": 0.001,
        "file": MODEL_FILES["CNN4"],
        "architecture": "Conv + BatchNorm + ReLU + MaxPool + Dropout; Linear + Linear"
    },
}

RUN_TRAINING = True
MODELS_TO_RUN = ["ResNet", "CNN1", "CNN2", "CNN3", "CNN4"]

## 7. Funções de treino e gráficos de aprendizagem

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in dataloader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels).item()

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_corrects / len(dataloader.dataset)
    return epoch_loss, epoch_acc

def validate_one_epoch(model, dataloader, criterion):
    model.eval()
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels).item()

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_corrects / len(dataloader.dataset)
    return epoch_loss, epoch_acc

def plot_history(history_df, model_name):
    fig = plt.figure(figsize=(8, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.title(f"Loss — {model_name}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / f"{model_name}_loss.png", dpi=150)
    plt.show()

    fig = plt.figure(figsize=(8, 4))
    plt.plot(history_df["epoch"], history_df["train_accuracy"], label="train_accuracy")
    plt.plot(history_df["epoch"], history_df["val_accuracy"], label="val_accuracy")
    plt.title(f"Accuracy — {model_name}")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / f"{model_name}_accuracy.png", dpi=150)
    plt.show()

def train_model(model_name, train_dl, val_dl):
    config = MODEL_CONFIGS[model_name]
    model = config["constructor"]().to(DEVICE)

    epochs = config["epochs"]
    lr = config["lr"]
    criterion = CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=lr)

    history = []
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_dl, criterion, optimizer)
        val_loss, val_acc = validate_one_epoch(model, val_dl, criterion)

        row = {
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        }
        history.append(row)

        print(
            f"[{model_name}] Epoch {epoch:03d}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    elapsed = time.perf_counter() - start
    print(f"Tempo gasto em {model_name}: {elapsed:.2f} segundos")

    torch.save(model, config["file"])

    history_df = pd.DataFrame(history)
    history_df.to_csv(RESULTS_DIR / f"{model_name}_history.csv", index=False)
    plot_history(history_df, model_name)

    return model, history_df

## 8. Treinar todos os modelos

Esta célula pode demorar. Em CPU, considera treinar um modelo de cada vez alterando `MODELS_TO_RUN`.

In [ ]:
trained_models = {}
histories = {}

if RUN_TRAINING:
    for model_name in MODELS_TO_RUN:
        print("\n" + "=" * 80)
        print("A treinar:", model_name)
        print("=" * 80)

        model, history_df = train_model(model_name, train_dl, val_dl)
        trained_models[model_name] = model
        histories[model_name] = history_df
else:
    print("RUN_TRAINING = False. O notebook vai usar os modelos .pth existentes para avaliação.")

## 9. Download/carregamento dos modelos `.pth`, se necessário

Usa esta célula se não treinaste os modelos e queres avaliar os `.pth` fornecidos.

In [ ]:
def download_pretrained_model(model_name):
    import gdown

    file_path = Path(MODEL_CONFIGS[model_name]["file"])
    if file_path.exists():
        print(f"{file_path} já existe.")
        return file_path

    file_id = GDRIVE_MODEL_IDS[model_name]
    url = f"https://drive.google.com/uc?id={file_id}"
    print(f"A descarregar {model_name} -> {file_path}")
    gdown.download(url, str(file_path), quiet=False)
    return file_path

def safe_torch_load(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

def get_model_for_evaluation(model_name):
    if model_name in trained_models:
        model = trained_models[model_name]
    else:
        file_path = Path(MODEL_CONFIGS[model_name]["file"])
        if not file_path.exists():
            download_pretrained_model(model_name)
        model = safe_torch_load(file_path)

    model = model.to(DEVICE)
    model.eval()
    return model

print("Modelos disponíveis localmente:")
for name, cfg in MODEL_CONFIGS.items():
    print(name, "->", Path(cfg["file"]).exists(), cfg["file"])

## 10. Avaliar os modelos

Para cada modelo:
- apresenta previsões;
- mostra accuracy e loss no teste;
- imprime `classification_report`;
- apresenta a matriz de confusão.

In [ ]:
def evaluate_model(dataloader, model, criterion=None):
    model.eval()
    predictions = []
    actual_values = []
    total_loss = 0.0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(inputs)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * inputs.size(0)

            preds = torch.argmax(outputs, dim=1)

            predictions.extend(preds.detach().cpu().numpy())
            actual_values.extend(labels.detach().cpu().numpy())

    predictions = np.array(predictions)
    actual_values = np.array(actual_values)

    acc = accuracy_score(actual_values, predictions)
    avg_loss = total_loss / len(dataloader.dataset) if criterion is not None else None

    return actual_values, predictions, acc, avg_loss

def display_predictions(actual_values, predictions, n=20):
    for i in range(min(n, len(actual_values))):
        real = int(actual_values[i])
        pred = int(predictions[i])
        status = "OK" if real == pred else "ERRO"
        print(f"{i:02d} | real={real} ({output_label(real)}) | previsão={pred} ({output_label(pred)}) | {status}")

def display_confusion_matrix(cm, list_classes, title):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(np.arange(len(list_classes)))
    ax.set_yticks(np.arange(len(list_classes)))
    ax.set_xticklabels(list_classes, rotation=45, ha="right")
    ax.set_yticklabels(list_classes)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / f"{title.replace(' ', '_')}.png", dpi=150)
    plt.show()

all_results = {}
criterion = CrossEntropyLoss()
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

for model_name in MODELS_TO_RUN:
    print("\n" + "=" * 80)
    print("Avaliação:", model_name)
    print("=" * 80)

    model = get_model_for_evaluation(model_name)
    actual_values, predictions, test_acc, test_loss = evaluate_model(test_dl, model, criterion)

    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")
    print("\nPrimeiras previsões:")
    display_predictions(actual_values, predictions, n=20)

    print("\nClassification report:")
    print(classification_report(actual_values, predictions, target_names=class_names))

    cm = confusion_matrix(actual_values, predictions)
    display_confusion_matrix(cm, class_names, f"Matriz de Confusão - {model_name}")

    all_results[model_name] = {
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "actual_values": actual_values,
        "predictions": predictions,
    }

## 11. Usar os modelos para prever um caso

Altera `CASE_INDEX` para experimentar outra imagem do batch de teste.

In [ ]:
def make_prediction(model, img):
    model.eval()
    img_batch = img.reshape(1, 3, 32, 32).to(DEVICE)

    with torch.no_grad():
        outputs = model(img_batch)
        prediction = torch.argmax(outputs, dim=1).item()

    return prediction

CASE_INDEX = 3

test_images, test_labels = next(iter(test_dl))
img = test_images[CASE_INDEX]
real_label = int(test_labels[CASE_INDEX])

plt.figure(figsize=(3, 3))
plt.axis("off")
plt.imshow(tensor_to_image(img))
plt.title(f"Real: {output_label(real_label)}")
plt.show()

print("Label real:", real_label, "-", output_label(real_label))
print()

for model_name in MODELS_TO_RUN:
    model = get_model_for_evaluation(model_name)
    pred = make_prediction(model, img)
    print(f"{model_name}: previsão={pred} - {output_label(pred)}")

## 12. Comparação final e melhor modelo

Esta tabela é o resumo que deves comentar na entrega.

In [ ]:
summary_rows = []

for model_name in MODELS_TO_RUN:
    cfg = MODEL_CONFIGS[model_name]
    row = {
        "model": model_name,
        "architecture": cfg["architecture"],
        "epochs": cfg["epochs"],
        "batch_size": BATCH_SIZE,
        "learning_rate": cfg["lr"],
        "loss_function": "CrossEntropyLoss",
        "optimizer": "SGD",
    }

    if model_name in histories:
        h = histories[model_name]
        best_val = h.loc[h["val_accuracy"].idxmax()]
        last = h.iloc[-1]
        row.update({
            "best_val_accuracy": best_val["val_accuracy"],
            "best_val_loss": best_val["val_loss"],
            "last_train_accuracy": last["train_accuracy"],
            "last_train_loss": last["train_loss"],
        })

    if model_name in all_results:
        row.update({
            "test_accuracy": all_results[model_name]["test_accuracy"],
            "test_loss": all_results[model_name]["test_loss"],
        })

    summary_rows.append(row)

results_df = pd.DataFrame(summary_rows)
display(results_df)

results_df.to_csv(RESULTS_DIR / "comparacao_modelos.csv", index=False)

if "test_accuracy" in results_df.columns:
    best_idx = results_df["test_accuracy"].astype(float).idxmax()
    best_row = results_df.loc[best_idx]
    print(f"Melhor modelo no teste: {best_row['model']} | accuracy={best_row['test_accuracy']:.4f}")

## 13. Gerar um ficheiro ZIP para entrega

Depois de executares o notebook todo, corre esta célula para criar um zip com:
- resultados `.csv`;
- gráficos;
- modelos `.pth` gerados/carregados;
- este notebook, se o nome for detetado no diretório atual.

No Colab, o notebook pode não aparecer automaticamente como ficheiro `.ipynb`; nesse caso faz primeiro `File > Download > Download .ipynb` e junta-o manualmente ao zip.

In [ ]:
def create_submission_zip(zip_name="entrega_CNN_RGB.zip"):
    files_to_zip = []

    # Resultados
    for p in RESULTS_DIR.glob("*"):
        if p.is_file():
            files_to_zip.append(p)

    # Modelos
    for cfg in MODEL_CONFIGS.values():
        p = Path(cfg["file"])
        if p.exists():
            files_to_zip.append(p)

    # Notebooks no diretório atual
    for p in Path(".").glob("*.ipynb"):
        files_to_zip.append(p)

    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files_to_zip:
            z.write(p, arcname=str(p))

    print(f"ZIP criado: {zip_name}")
    print("Ficheiros incluídos:")
    for p in files_to_zip:
        print("-", p)

create_submission_zip()